In [ ]:
# ============================================================
# CELL 13: Enhanced Strategy Implementation RSI + MACD
# ============================================================
#
# IMPLEMENT YOUR ENHANCED STRATEGY HERE
#
# This is where you will implement improvements to the baseline strategy.
# Consider the five exploration areas:
#
# Area 1: Enhanced EDA
#   - Additional visualizations for insights (use Cell 12)
#
# Area 2: Data Quality & Cleaning (override clean_data)
#   - Forward-fill missing prices
#   - Filter outliers
#   - Remove tickers with insufficient data
#
# Area 3: More Technical Indicators (override calculate_analytics)
#   - RSI: 14-period relative strength index
#   - MACD: Moving Average Convergence Divergence
#   - Bollinger Bands: 20-day +-2 standard deviations
#
# Area 4: Enhanced LLM Analysis (override llm_analysis)
#   - Use more text from transcripts (4000+ characters)
#   - Calculate sentiment strength (positive - negative scores)
#   - Weight decisions by confidence levels
#
# Area 5: Smarter Decision Logic (override make_decision)
#   - Multi-signal confirmation (require 2+ signals)
#   - Position sizing based on signal strength
#   - Stop-loss / take-profit mechanisms
#   - Risk management controls
#
# Expected Improvements:
#   - Higher Total Return (target: >50%)
#   - Higher Sharpe Ratio (target: >1.0)
#   - Lower Max Drawdown (target: <30%)
#   - Lower Volatility (target: <40%)
#   - Higher Win Rate (target: >40%)
import pandas as pd
import numpy as np
import time
import warnings
from datetime import timedelta
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

print("\n" + "="*70)
print("ENHANCED STRATEGY")
print("="*70)

class EnhancedStrategy(BaseStrategy):
    def __init__(self, finbert_pipeline=None):
        """Initialize your enhanced strategy."""
        # Initialize any strategy-specific attributes here
        # Example: self.entry_dates = {}  # For holding period tracking
        super().__init__(finbert_pipeline)
        # For Area 5: Tracking the "Weekly Ratchet" for each ticker
        self.highest_price_seen = {}

    def clean_data(self, prices_df, earnings_df):
        """
        Override to implement enhanced data cleaning.

        Ideas:
        - Forward-fill missing prices by ticker
        - Filter tickers with insufficient data
        - Remove outliers
        """
        # 1. Sort and Forward-fill to maintain weekly continuity
        prices_df = prices_df.sort_values(['ticker', 'date'])
        prices_df[['open', 'high', 'low', 'close']] = prices_df.groupby('ticker')[['open', 'high', 'low', 'close']].ffill()

        # 2. Minimum Price Filter ($5) to avoid 'penny stock' volatility
        prices_df = prices_df[prices_df['close'] >= 5.0]

        # 3. History Filter: Ensure 200 days exist for stable EMA-200 calculation
        ticker_counts = prices_df.groupby('ticker').size()
        valid_tickers = ticker_counts[ticker_counts >= 200].index
        prices_df = prices_df[prices_df['ticker'].isin(valid_tickers)]
        return prices_df, earnings_df

    def calculate_analytics(self, prices_df):
        """
        Override to compute additional technical indicators.

        Ideas:
        - RSI: 14-period relative strength index
        - MACD: EMA-12, EMA-26, signal line
        - Bollinger Bands: 20-day +-2 std dev

        Consider: How can you parallelize or optimize these computations
        to handle 400+ tickers efficiently?
        """
        """Area 3: Multi-Layered Technical Indicators."""
        # Using vectorized operations for speed (crucial for 400+ tickers)
        # Structural Trend
        prices_df['ema_200'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=200, adjust=False).mean())
        prices_df['ema_50'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=50, adjust=False).mean())

        # RSI (14-period)
        prices_df['rsi'] = prices_df.groupby('ticker')['close'].transform(self._calculate_rsi, window=14)

        #MACD (12, 26, 9)
        prices_df['ema_12'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
        prices_df['ema_26'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
        prices_df['macd'] = prices_df['ema_12'] - prices_df['ema_26']
        prices_df['signal_line'] = prices_df.groupby('ticker')['macd'].transform(lambda x: x.ewm(span=9, adjust=False).mean())
        prices_df['macd_hist'] = prices_df['macd'] - prices_df['signal_line']

        # Pace Picking up (Histogram Slope)
        prices_df['hist_prev'] = prices_df.groupby('ticker')['macd_hist'].shift(1)
        prices_df['pace_improving'] = prices_df['macd_hist'] > prices_df['hist_prev']

        # Convert dates to strings for comparison (required by TradingSimulation)
        prices_df['date'] = prices_df['date'].dt.strftime('%Y-%m-%d')
        
        return prices_df

    def _calculate_rsi(self, prices, window):
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def llm_analysis(self, ticker, transcript, date):
        """
        Override to enhance NLP analysis.

        Ideas:
        - Use more text (4000+ chars)
        - Calculate sentiment strength (positive - negative)
        - Weight decisions by confidence
        """
        """Area 4: Enhanced NLP Analysis with SafeTensors."""
        # Use the first 4000 characters to capture CEO's speech beyond the operator intro
        if not transcript or len(transcript) < 100:
            return None  # No transcript this week — don't penalise entry
        
        try:
            sentiment = self.finbert_pipeline(transcript[:4000], truncation=True)[0]
            score_map = {'positive': 1.0, 'neutral': 0.5, 'negative': 0.0}
            return score_map.get(sentiment['label'], 0.5)
        except Exception as e:
            print(f"Error occurred while analyzing sentiment for {ticker}: {e}")
            return None  # On error, don't block the trade

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        """
        Override to implement smarter trading logic.

        Ideas:
        - Combine multiple indicators (RSI, MACD, sentiment)
        - Require 2+ signals for entry/exit
        - Add stop-loss / take-profit
        - Implement minimum holding period
        """
        """Area 5: Smarter Decision Logic - Structural Rebound."""
        curr_price = analytics.get('close', 0)
        if not curr_price:
            return 'HOLD'

        # --- EXIT LOGIC---
        if ticker in portfolio_state['positions']:
            #Update Weekly Ratchet
            self.highest_price_seen[ticker] = max(self.highest_price_seen.get(ticker, 0), curr_price)
            
            # 1. Trailing Stop-Loss (-15% from Friday Peak)
            stop_price = self.highest_price_seen[ticker] * 0.85
            if curr_price < stop_price:
                del self.highest_price_seen[ticker]
                return 'SELL'
            
            # 2. Momentum Decay (Histogram turned down)
            if not analytics.get('pace_improving', True):
                del self.highest_price_seen[ticker]
                return 'SELL'
        
        # --- ENTRY LOGIC ---
        # 1. Structural Trend: Price above EMA-200 and EMA-50
        structural_ok = curr_price > analytics.get('ema_50', float('inf')) and curr_price > analytics.get('ema_200', float('inf'))

        # 2. Momentum Reversal: RSI < 55 (not overbought) and MACD Histogram is rising
        momentum_rebound = analytics.get('rsi', 100) < 45 and analytics.get('pace_improving', False)

        # 3. Sentiment Confirmation: only block if sentiment is explicitly negative
        sentiment_score = self.llm_analysis(ticker, transcript, date)
        sentiment_ok = sentiment_score is None or sentiment_score >= 0.5

        if structural_ok and momentum_rebound and sentiment_ok:
            # Initialize the highest price seen for this ticker
            self.highest_price_seen[ticker] = curr_price
            return 'BUY'
        
        return 'HOLD'
    '''
        Note: While this implementation focuses on overriding the core strategy methods
        (clean_data, calculate_analytics, llm_analysis, make_decision), you may optionally
        override other base class methods such as evaluate() to implement custom evaluation
        logic or you can add new functions to this class, provided that all modifications
        maintain backward compatibility and execute without runtime errors.
    '''

print("EnhancedStrategy class loaded (implement methods above to improve strategy)")

In [ ]:
# ============================================================
# CELL 13: Enhanced Strategy Implementation RSI + MACD + BOLLINGER
# ============================================================
import pandas as pd
import numpy as np
import time
import warnings
from datetime import timedelta
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

print("\n" + "="*70)
print("ENHANCED STRATEGY: VOLATILITY-ADJUSTED REBOUND")
print("="*70)

class EnhancedStrategy(BaseStrategy):
    def __init__(self, finbert_pipeline=None):
        super().__init__(finbert_pipeline)
        self.highest_price_seen = {}

    def clean_data(self, prices_df, earnings_df):
        prices_df = prices_df.sort_values(['ticker', 'date'])
        prices_df[['open', 'high', 'low', 'close']] = prices_df.groupby('ticker')[['open', 'high', 'low', 'close']].ffill()
        prices_df = prices_df[prices_df['close'] >= 5.0]
        ticker_counts = prices_df.groupby('ticker').size()
        valid_tickers = ticker_counts[ticker_counts >= 200].index
        prices_df = prices_df[prices_df['ticker'].isin(valid_tickers)]
        return prices_df, earnings_df

    def calculate_analytics(self, prices_df):
        # 1. Structural Trends
        prices_df['ema_200'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=200, adjust=False).mean())
        prices_df['ema_50'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=50, adjust=False).mean())

        # 2. Rebound Indicators (RSI & MACD)
        prices_df['rsi'] = prices_df.groupby('ticker')['close'].transform(self._calculate_rsi, window=14)
        prices_df['ema_12'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
        prices_df['ema_26'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
        prices_df['macd_hist'] = (prices_df['ema_12'] - prices_df['ema_26']) - (prices_df['ema_12'] - prices_df['ema_26']).ewm(span=9).mean()
        prices_df['pace_improving'] = prices_df['macd_hist'] > prices_df.groupby('ticker')['macd_hist'].shift(1)

        # 3. Bollinger Bands (The Floor)
        prices_df['sma_20'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=20).mean())
        prices_df['std_20'] = prices_df.groupby('ticker')['close'].transform(lambda x: x.rolling(window=20).std())
        prices_df['bb_lower'] = prices_df['sma_20'] - (prices_df['std_20'] * 2)

        return prices_df

    def _calculate_rsi(self, prices, window):
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
        rsi = 100 - (100 / (1 + (gain / loss)))
        return rsi

    def llm_analysis(self, ticker, transcript, date):
        if not transcript or len(transcript) < 100: return None
        try:
            sentiment = self.finbert_pipeline(transcript[:4000], truncation=True)[0]
            return {'positive': 1.0, 'neutral': 0.5, 'negative': 0.0}.get(sentiment['label'], 0.5)
        except: return None

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        curr_price = analytics.get('close', 0)
        if not curr_price: return 'HOLD'

        # --- EXIT LOGIC (Volatility Adjusted) ---
        if ticker in portfolio_state['positions']:
            self.highest_price_seen[ticker] = max(self.highest_price_seen.get(ticker, 0), curr_price)
            
            # Dynamic Stop: Price - (2 * ATR) instead of a flat 15%
            # This protects you more in stable markets and gives room in volatile ones
            vol_stop = self.highest_price_seen[ticker] - (2 * analytics.get('atr', curr_price * 0.05))
            
            if curr_price < vol_stop or not analytics.get('pace_improving', True):
                del self.highest_price_seen[ticker]
                return 'SELL'

        # --- ENTRY LOGIC (The original Bollinger Rebound + Trend Strength) ---
        structural_ok = curr_price > analytics.get('ema_50', 0) and curr_price > analytics.get('ema_200', 0)
        near_floor = curr_price <= (analytics.get('bb_lower', 0) * 1.06) # 6% buffer
        momentum_rebound = analytics.get('rsi', 100) < 45 and analytics.get('pace_improving', False)
        
        # Only enter if the 'Trend Strength' is starting to turn positive
        trend_ok = analytics.get('trend_strength', 0) > -1.0 

        sentiment_score = self.llm_analysis(ticker, transcript, date)
        sentiment_ok = sentiment_score is None or sentiment_score >= 0.5

        if structural_ok and near_floor and momentum_rebound and trend_ok and sentiment_ok:
            self.highest_price_seen[ticker] = curr_price
            return 'BUY'
        
        return 'HOLD'

print("EnhancedStrategy loaded with ATR Volatility Stops & Trend Strength.")